# Ollama → backend stream

Run the Go backend locally (`go run .` from `backend/`). It tunnels a WebSocket URL into the injection cell below.

Run this notebook on Kaggle: installs Ollama + **tinyllama**, connects to your machine, streams two LLM replies, then exits on `DONE`.

In [ ]:
# Injection template
BACKEND_WS_URL = ""

In [ ]:
# Injection space


In [ ]:
!pip install -q websockets

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import asyncio, json, subprocess, time, urllib.request, websockets

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)
subprocess.run(["ollama", "pull", "tinyllama"])

def ollama_chunks(prompt):
    req = urllib.request.Request(
        "http://localhost:11434/api/generate",
        data=json.dumps({"model": "tinyllama", "prompt": prompt, "stream": True}).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req) as resp:
        for line in resp:
            part = json.loads(line).get("response", "")
            if part:
                yield part

async def run():
    async with websockets.connect(BACKEND_WS_URL) as ws:
        await ws.send("READY")
        while True:
            msg = await ws.recv()
            if msg == "DONE":
                break
            for chunk in ollama_chunks(msg):
                await ws.send(chunk)
            await ws.send("END")

await run()